# Chunking comparison notebook

This notebook compares two chunking strategies side-by-side on representative documents from the knowledge base:

- RecursiveCharacterTextSplitter with chunk size 800 and overlap 100
- Semantic chunking using the project helper

In [1]:
import sys
from pathlib import Path
from typing import Dict, List, Tuple

# Ensure project root is on sys.path
# Walk up from CWD looking for a sentinel file (requirements.txt or .git)
_candidate = Path.cwd().resolve()
for _ in range(10):
    if (_candidate / 'requirements.txt').exists() or (_candidate / '.git').exists():
        project_root = _candidate
        break
    _candidate = _candidate.parent
else:
    # Fallback: assume notebook is in notebooks/ subdirectory of project
    project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display, Markdown

from app.rag.loaders import load_documents_from_manifest
from app.rag.chunkers import ChunkConfig, chunk_document

In [2]:
documents = load_documents_from_manifest()

representative_docs = {
    "policy_wording": next(doc for doc in documents if doc.doc_type == "policy_wording"),
    "regulation": next(doc for doc in documents if doc.doc_type == "regulation"),
    "memo": next(doc for doc in documents if doc.doc_type == "memo"),
}

representative_docs

{'policy_wording': <app.rag.loaders.Document at 0x717bd23b5430>,
 'regulation': <app.rag.loaders.Document at 0x717a0c3037c0>,
 'memo': <app.rag.loaders.Document at 0x717a0c38c100>}

In [3]:
def compare_chunking(doc):
    config = ChunkConfig(chunk_size=800, chunk_overlap=100)
    recursive_chunks = chunk_document(doc, config, use_semantic=False)
    semantic_chunks = chunk_document(doc, config, use_semantic=True)
    return recursive_chunks, semantic_chunks

def preview_text(text: str, limit: int = 220) -> str:
    cleaned = " ".join(text.split())
    return cleaned[:limit] + ("..." if len(cleaned) > limit else "")

def build_side_by_side_table(doc, max_rows: int = 6):
    recursive_chunks, semantic_chunks = compare_chunking(doc)
    rows = []
    for idx in range(max(len(recursive_chunks), len(semantic_chunks))):
        recursive_chunk = recursive_chunks[idx] if idx < len(recursive_chunks) else None
        semantic_chunk = semantic_chunks[idx] if idx < len(semantic_chunks) else None
        rows.append({
            "chunk_index": idx,
            "recursive_preview": preview_text(recursive_chunk.text) if recursive_chunk else "",
            "semantic_preview": preview_text(semantic_chunk.text) if semantic_chunk else "",
        })
    return pd.DataFrame(rows).head(max_rows)

summary_rows = []
for label, doc in representative_docs.items():
    recursive_chunks, semantic_chunks = compare_chunking(doc)
    summary_rows.append({
        "document": label,
        "source_id": doc.source_id,
        "doc_type": doc.doc_type,
        "recursive_chunks": len(recursive_chunks),
        "semantic_chunks": len(semantic_chunks),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,document,source_id,doc_type,recursive_chunks,semantic_chunks
0,policy_wording,health_policy_hdfcergo,policy_wording,195,296
1,regulation,irda_health_reg_2016,regulation,98,403
2,memo,prior_adjudication_memos_0,memo,1,1


In [4]:
for label, doc in representative_docs.items():
    display(Markdown(f"## {label}: {doc.source_id}\n\n**doc_type**: {doc.doc_type}"))
    display(build_side_by_side_table(doc))

## policy_wording: health_policy_hdfcergo

**doc_type**: policy_wording

,chunk_index,recursive_preview,semantic_preview
0,0,HDFC ERGO General Insurance Company Limited Po...,HDFC ERGO General Insurance Company Limited
1,1,"wherever they appear in this Policy and, where...",Policy Wording Easy Health HDFC ERGO General I...
2,2,procedures and interventions are carried out b...,1 Preamble HDFC ERGO General Insurance Company...
3,3,ii. Having qualified AYUSH Medical Practitione...,violent means. Def. 2 Any one illness means co...
4,4,and medical or surgical/para -surgical interve...,"c. AYUSH Hospital, standalone or co-located wi..."
5,5,HDFC ERGO General Insurance Company Limited Po...,Def. 4 AYUSH Day Care Centre means and include...


## regulation: irda_health_reg_2016

**doc_type**: regulation

,chunk_index,recursive_preview,semantic_preview
0,0,EXPOSURE DRAFT ON INSURANCE REGULATORY AND DEV...,EXPOSURE DRAFT
1,1,"Unless otherwise specified herein, these Regul...",ON
2,2,“Break in policy” means the period of gap that...,INSURANCE REGULATORY AND DEVELOPMENT AUTHORITY...
3,3,“Health insurance business”means the effecting...,F. No. IRDA/Reg/…./…/2016 - In exercise of the...
4,4,“Pilot product” means a close-ended product ru...,CHAPTER I - GENERAL
5,5,“Third Party Administrators or TPA means any p...,Short title and commencement.


## memo: prior_adjudication_memos_0

**doc_type**: memo

,chunk_index,recursive_preview,semantic_preview
0,0,Memo Id: MEMO001 Claim Id: CLM2025001 Member I...,Memo Id: MEMO001 Claim Id: CLM2025001 Member I...
